# Random Forest Classifier Evaluation

In [7]:
import sys; sys.path.append("..")
import src.preprocessing as pp

# Random Forest Class

In [8]:
# Random forest baseline trained on soft (vote-share) labels.
import numpy as np
from sklearn.ensemble import RandomForestClassifier

from src.evaluation import CLASS_NAMES, RANDOM_STATE
from src.models import expand_soft_labels

N_CLASSES = len(CLASS_NAMES)


class SoftLabelRandomForest:
    """Random forest that learns from vote distributions and predicts them.

    fit(X, y): X is preprocessed features, y is (n_segments, 6) vote shares.
    predict_proba(X): returns (n_segments, 6) probabilities, columns in
    CLASS_NAMES order, each row summing to 1.
    """

    def __init__(self, n_estimators=300, min_samples_leaf=5, max_features="sqrt",
                 random_state=RANDOM_STATE, n_jobs=-1, **rf_kwargs):
        self.forest = RandomForestClassifier(
            n_estimators=n_estimators,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=random_state,
            n_jobs=n_jobs,
            **rf_kwargs,
        )

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        # Drop segments with no votes (all-zero rows) so they carry no weight.
        keep = y.sum(axis=1) > 0
        X = X.iloc[np.flatnonzero(keep)] if hasattr(X, "iloc") else np.asarray(X)[keep]
        y = y[keep]
        # Renormalise so each segment's weights sum to 1.
        y = y / y.sum(axis=1, keepdims=True)

        X_exp, classes, weights = expand_soft_labels(X, y)
        self.forest.fit(X_exp, classes, sample_weight=weights)
        return self

    def predict_proba(self, X):
        raw = self.forest.predict_proba(X)
        # Early active-learning rounds may not have seen every class.
        # Place the learned columns in the full 6-class layout.
        proba = np.zeros((raw.shape[0], N_CLASSES))
        proba[:, self.forest.classes_.astype(int)] = raw
        return proba


def fit_predict_rf(X_fit, y_fit, X_val, **rf_kwargs):
    """Signature expected by evaluation.cross_validate."""
    model = SoftLabelRandomForest(**rf_kwargs).fit(X_fit, y_fit)
    return model.predict_proba(X_val)

# Code Usage

In [10]:
import pandas as pd

# Shared helpers from the project package:
# - get_cv_folds: builds the 5 patient-grouped cross-validation folds
# - cross_validate: trains/scores a model on those folds
# - evaluate: computes KL divergence and macro-F1 for one set of predictions
from src.evaluation import get_cv_folds, cross_validate, evaluate

# - get_feature_cols: finds the EEG/spectrogram feature columns by name prefix
# - get_soft_labels: converts vote counts into vote shares (each row sums to 1)
# - preprocess: imputes, transforms and scales features, fitted on train only
# - load_selected_features: reads the team's agreed 30-feature list from JSON
from src.preprocessing import (get_feature_cols, get_soft_labels, preprocess,
                            load_selected_features)


# --- Load (same tables as the notebook) ---

# Folder containing the prepared CSV files. Change to your own path.
data_dir = "../HMSData_prepared"

# One row per segment: patient ID plus the precomputed EEG/spectrogram features
features = pd.read_csv(f"{data_dir}/features.csv")

# One row per segment: expert vote counts per class, num_votes, expert_consensus
labels   = pd.read_csv(f"{data_dir}/label.csv")

# One row per segment: which group it belongs to (train_pool, test, or excluded)
split    = pd.read_csv(f"{data_dir}/train_test.csv")

# Take only the 'group' column from split. patient_id and num_votes already
# exist in labels, so including them would create duplicate columns.
# validate="one_to_one" raises an error if any segment_id appears twice.
meta = labels.merge(split[["segment_id", "group"]], on="segment_id",
                    validate="one_to_one")

# Attach the features to the labels and group. Merging on both keys avoids a
# duplicated patient_id column, since features.csv also contains it.
data = features.merge(meta, on=["segment_id", "patient_id"],
                      validate="one_to_one")

# Training pool: patients with enough votes, used to fit the model.
# reset_index gives clean 0..n-1 row numbers, which the CV folds rely on.
train_data = data[data["group"] == "train_pool"].reset_index(drop=True)

# Held-out test set: never used for fitting, preprocessing or feature selection
test_data  = data[data["group"] == "test"].reset_index(drop=True)

# --- Preprocess (fit on the training pool only) ---

# Names of the feature columns (eeg_* and spect_*). Metadata columns such as
# patient_id and the vote counts are excluded automatically.
feature_cols = get_feature_cols(data)

# Learn the imputation medians, skew transform and scaling from train only,
# then apply those same values to test. This prevents test information from
# leaking into the model.
X_train_scaled, X_test_scaled = preprocess(train_data[feature_cols],
                                           test_data[feature_cols])

# The 30 features the team chose by importance ranking
selected_features = load_selected_features()

# Keep only those 30 columns, giving shapes (5569, 30) and (1155, 30)
X_train_final = X_train_scaled[selected_features]
X_test_final  = X_test_scaled[selected_features]

# Soft labels: each row is the vote share per class, in the order
# Seizure, LPD, GPD, LRDA, GRDA, Other. The target for the forest, and the
# reference for KL and macro-F1.
y_train = get_soft_labels(train_data)
y_test  = get_soft_labels(test_data)

# Patient ID per training segment. The CV split uses it to keep every patient
# in exactly one fold.
groups  = train_data["patient_id"].to_numpy()

# --- Train on the training pool, evaluate on the held-out test set ---

# Create the random forest and fit it on the training features and soft labels
rf = SoftLabelRandomForest().fit(X_train_final, y_train)

# Predicted probabilities for each test segment: shape (1155, 6), columns in
# CLASS_NAMES order, each row summing to 1
test_probs = rf.predict_proba(X_test_final)

# Compare the predictions with the experts' vote shares. Prints a dict with:
# - "kl": mean KL divergence (lower is better, 0 is perfect)
# - "macro_f1": macro-F1 of the predicted top class vs. the majority vote
print(evaluate(y_test, test_probs))

# --- Shared patient-grouped 5-fold CV on the training pool ---

# Build the 5 folds. They are stratified by majority class and grouped by
# patient, so everyone on the team gets identical splits.
folds = get_cv_folds(y_train, groups)

# For each fold: train on the other 4 folds, predict the held-out fold, and
# score it. Returns a DataFrame with one row per fold (fold, kl, macro_f1).
cv_results = cross_validate(fit_predict_rf, X_train_final, y_train, folds)

# Per-fold scores, then the average over the 5 folds
print(cv_results, cv_results[["kl", "macro_f1"]].mean())

{'kl': np.float64(0.48792006808801003), 'macro_f1': 0.428043818677905}
   fold        kl  macro_f1
0     0  0.470871  0.416820
1     1  0.455941  0.483095
2     2  0.485943  0.459894
3     3  0.463954  0.462873
4     4  0.546045  0.411871 kl          0.484551
macro_f1    0.446911
dtype: float64
